# Week 5 Exercise : Private Knowledge Worker

1. Assemble all your files in 1 place; your personal Knowledge Base
2. Vectorize everything in Chroma - your vector datastore
3. Build a Conversational AI and ask questions!

In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

from sklearn.manifold import TSNE
import plotly.graph_objects as go

import gradio as gr

In [ ]:

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


In [ ]:
knowledge_base_path = "Books/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

In [ ]:
entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"


In [ ]:
#Encoding
encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)

In [ ]:
# Load in everything in the knowledgebase using LangChain's loaders
folder = "Books/"  # the only folder containing .md files
doc_type = os.path.basename(folder)
loader = DirectoryLoader(folder, glob="*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
documents = []

folder_docs = loader.load()
for doc in folder_docs:
    doc.metadata["doc_type"] = doc_type  # optional: label docs with their folder
    documents.append(doc)

print(f"Loaded {len(documents)} documents")

In [ ]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)


In [ ]:
# Pick an embedding model

# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)

In [9]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [10]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant that helps with enquiries from a personal knowledge base.
You are chatting with a user about the knowledge base.
If relevant, use the given context to answer any question.
Only include details from the context that are relevant to the question. And only if the question or the context demands it.

If the question is about scientific or technical topics, state the source of the information and facts. 
You are allowed to hypothesize but you are not allowed to make up facts. 
If you hypothesize, state that you are hypothesizing.
If you make assumptions, state that you are assuming.
If you don't know the answer, say so.

Context:
{context}
"""

In [11]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
gr.ChatInterface(answer_question, title="Personal Knowledge Worker", description="Ask questions about your knowledge base").launch(inbrowser=True)